This is the Update file for the heroes table. All you need to do is run all cells to update the table and hero-pages folder to include the most recent data from Fire Emblem Heroes.

In [1]:
import pandas as pd
import bs4
import requests
import plotly.express as px
import kaleido
import regex
import time
import os
import lxml

In [ ]:
heroes = requests.get('https://feheroes.fandom.com/wiki/Level_40_stats_table')

In [ ]:
heroes = bs4.BeautifulSoup(heroes.text)

In [ ]:
rows = heroes.find_all('tr', attrs={'class':'hero-filter-element'})
# Gets all rows of Heroes as a list


In [ ]:
links = [] # This cell creates the links for each individual heroes' info pages
for i in range(len(rows)):
    links.append("https://feheroes.fandom.com" + rows[i].find_all('a')[0].get('href'))
links

In [ ]:
heroes_table = pd.DataFrame(columns=['Hero', 'Entry', 'Move', 'Weapon', 'HP', 'Atk', 'Spd', 'Def', 'Res', 'Total'])
for i in range(len(rows)):
    row = rows[i].find_all('td')
    heroes_table.loc[len(heroes_table)] = [
        row[0].find('a').get('title'),
        row[2].find('img').get('alt'),
        row[3].find('img').get('alt'),
        row[4].find('img').get('alt'),
        int(row[5].text),
        int(row[6].text),
        int(row[7].text),
        int(row[8].text),
        int(row[9].text),
        int(row[10].text)
    ]
heroes_table['Link'] = links
heroes_table # This heroes table only includes data that is available on the Lv. 40 Stats Table on the Fandom page

In [ ]:
def color(str):
    return str.split(' ')[0]

heroes_table['Color'] = heroes_table['Weapon'].apply(color)
heroes_table.head()
# This bit just adds the color of the unit by pulling from the Weapon column

Next is adding book information by pulling data from hero pages stored in the hero-pages directory.

In [ ]:
test = bs4.BeautifulSoup(requests.get(heroes_table['Link'].iloc[1]).text)
float(test.find('table', attrs={'class':'wikitable hero-infobox'}).find_all('a')[-1].text)
# Test is here to find where book info is stored.

In [ ]:
# SoupStrainer makes the process WAY FASTER
only_table = bs4.SoupStrainer('table')
with open(f'hero-pages/AbelThePanther.html', 'r', encoding="UTF-8") as f:
    soup = bs4.BeautifulSoup(f.read(), 'lxml', parse_only=only_table)
    print(float(soup.find('table', attrs={'class':'wikitable hero-infobox'}).find_all('a')[-1].text))

In [ ]:
versions = []
only_table = bs4.SoupStrainer('table')
for i in range(heroes_table.shape[0]):
    temp = heroes_table['Hero'].iloc[i].split()
    for j in range(len(temp)):
        temp[j] = temp[j].strip(':"')
        name = ''.join(temp)
        print(name)
    
    with open(f'hero-pages/{name}.html', 'r', encoding="UTF-8") as f:
        soup = bs4.BeautifulSoup(f.read(), 'lxml', parse_only=only_table)

In [ ]:
# Add column
heroes_table['Version'] = versions
heroes_table.head()

In [ ]:
# 47m 3.2s runtime

# def version(link):
#     time.sleep(0.5)
#     html = bs4.BeautifulSoup(requests.get(link).text)
#     return float(html.find('table', attrs={'class':'wikitable hero-infobox'}).find_all('a')[-1].text)

# heroes_table['Version'] = heroes_table['Link']
# heroes_table['Version'] = heroes_table['Version'].apply(version)

# Obsolete code. Very inefficient way to pull version info for each unit for Version Column

In [ ]:
# heroes_pages = heroes_table[['Hero', 'Link']]
# heroes_pages.head()
# Hero and Link table

In [ ]:
# runtime 50m 5.8s

# def grabber(link):
#     time.sleep(0.5)
#     print(link.split('/')[-1])
#     return bs4.BeautifulSoup(requests.get(link).text)

# heroes_pages['Pages'] = heroes_pages['Link'].apply(grabber)

# Another obsolete bit of code that essentially adds the entire HTML page of each hero in a Page column.

In [ ]:
# heroes_table = heroes_table.drop(columns=['Link'])
heroes_table.to_csv('heroes_table.csv') # I'll keep the link column in, just as a reference.

##### Below is code that essentially pulls each hero's info page as an HTML file and adds it to the hero-pages folder. This is done to keep the file size of the heroes table to a minimum, as well as keeping a local reference for each page that makes accessing these pages quicker and more efficient. The hero-pages folder is gitignored.

In [7]:
heroes_table = pd.read_csv('heroes_table.csv').drop(columns=['Unnamed: 0'])
pagedir = os.listdir('hero-pages') # Current hero pages in hero-pages directory

In [8]:
missingidx = []
for i in range(heroes_table.shape[0]):
    temp = heroes_table['Hero'].iloc[i].split()
    for j in range(len(temp)):
        temp[j] = temp[j].strip(':"')
        name = ''.join(temp)

    if f'{name}.html' not in pagedir:
        print(heroes_table['Hero'].iloc[i])
        missingidx.append(i)

# Use this to check which pages are currently not in directory

Rennac: Rich "Merchant"
Reyson: White Prince
Rhajat: Black Magician
Rhea: Aura of Love
Rhea: Immaculate One
Rhea: Loving Matriarch
Rhea: Witch of Creation
Rhys: Gentle Basker
Rickard: Carefree Culprit
Ricken: Shepherd Novice
Riev: Blood Beryl
Rinea: Reminiscent Belle
Rinkah: Consuming Flame
Rinkah: Scion of Flame
Robin: Exalt's Deliverer
Robin: Exalt's Other Half
Robin: Exalt's Right Hand
Robin: Fall Reincarnation
Robin: Fall Vessel
Robin: Fated Vessel
Robin: Fell Reincarnation
Robin: Fell Tactician
Robin: Fell Vessel
Robin: Festive Tactician
Robin: High Deliverer
Robin: Keen Groom
Robin: Mystery Tactician
Robin: Seaside Tactician
Robin: Tactful Deliverer
Robin: Vessels of Fate
Roderick: Steady Squire
Rolf: Tricky Archer
Ronan: Villager of Iz
Rosado: Adorable Artist
Roshea: Coyote's Faithful
Ross: His Father's Son
Roy: Blazing Bachelors
Roy: Blazing Lion
Roy: Brave Lion
Roy: Young Lion
Roy: Youthful Gifts
Rudolf: Emperor of Rigel
Rune: Source of Wisdom
Rutger: Lone Swordsman
Ryoma: Dan

The missingidx list created above can be used to add the missing hero pages to the hero-pages folder.

In [10]:
for i in missingidx:
    # Create file name
    temp = heroes_table['Hero'].iloc[i].split()
    for j in range(len(temp)):
        temp[j] = temp[j].strip(':"')
        name = ''.join(temp)
    
    # Pull HTML from site
    time.sleep(0.5)
    soup = bs4.BeautifulSoup(requests.get(heroes_table['Link'].iloc[i]).content, "lxml")
    script = soup.find_all('script')
    [s.decompose() for s in script]
    navbox = soup.find('div', class_= 'navbox')
    navbox.decompose()
    aside = soup.find_all('aside')
    [a.decompose() for a in aside]
    commhead = soup.find('div', class_='community-header-wrapper')
    commhead.decompose()
    ul = soup.find_all('ul', class_='wds-tabs')
    [u.decompose() for u in ul]
    foot = soup.find('footer', class_='global-footer')
    foot.decompose()
    pagef = soup.find('div', class_='page-footer')
    pagef.decompose()
    sidew = soup.find('div', class_='page-side-tools__wrapper')
    sidew.decompose()
    pheadt = soup.find('div', class_='page-header__top')
    pheadt.decompose()
    item1 = soup.find('div', class_='global-explore-navigation')
    item1.decompose()
    navtop = soup.find('div', class_='community-navigation fandom-sticky-header')
    navtop.decompose()
    ads = soup.find('div', class_='bottom-ads-container')
    ads.decompose()
    navi = soup.find('nav', class_='global-top-navigation')
    navi.decompose()
    act = soup.find('div', class_='page-header__actions')
    act.decompose()
    soup.find_all('table')[1].decompose()
    [svg.decompose() for svg in soup.find_all('svg')]
    soup.find('div', class_='toc').decompose()
    with open(F"hero-pages/{name}.html", "w", encoding = 'utf-8') as file:
        file.write(str(soup.prettify()))
    print(name)


RennacRichMerchant
ReysonWhitePrince
RhajatBlackMagician
RheaAuraofLove
RheaImmaculateOne
RheaLovingMatriarch
RheaWitchofCreation
RhysGentleBasker
RickardCarefreeCulprit
RickenShepherdNovice
RievBloodBeryl
RineaReminiscentBelle
RinkahConsumingFlame
RinkahScionofFlame
RobinExalt'sDeliverer
RobinExalt'sOtherHalf
RobinExalt'sRightHand
RobinFallReincarnation
RobinFallVessel
RobinFatedVessel
RobinFellReincarnation
RobinFellTactician
RobinFellVessel
RobinFestiveTactician
RobinHighDeliverer
RobinKeenGroom
RobinMysteryTactician
RobinSeasideTactician
RobinTactfulDeliverer
RobinVesselsofFate
RoderickSteadySquire
RolfTrickyArcher
RonanVillagerofIz
RosadoAdorableArtist
RosheaCoyote'sFaithful
RossHisFather'sSon
RoyBlazingBachelors
RoyBlazingLion
RoyBraveLion
RoyYoungLion
RoyYouthfulGifts
RudolfEmperorofRigel
RuneSourceofWisdom
RutgerLoneSwordsman
RyomaDancingSamurai
RyomaPeerlessSamurai
RyomaSamuraiatEase
RyomaSupremeSamurai
RyomaWarriorofBonds
SaberDrivenMercenary
SaberDuneWalker
SafyFontofPiety
S

# TESTING ZONE

In [ ]:
soup = bs4.BeautifulSoup(requests.get("https://feheroes.fandom.com/wiki/Soren:_Isolated_Strategist").content, "lxml")
script = soup.find_all('script')
[s.decompose() for s in script]
navbox = soup.find('div', class_= 'navbox')
navbox.decompose()
aside = soup.find_all('aside')
[a.decompose() for a in aside]
commhead = soup.find('div', class_='community-header-wrapper')
commhead.decompose()
ul = soup.find_all('ul', class_='wds-tabs')
[u.decompose() for u in ul]
foot = soup.find('footer', class_='global-footer')
foot.decompose()
pagef = soup.find('div', class_='page-footer')
pagef.decompose()
sidew = soup.find('div', class_='page-side-tools__wrapper')
sidew.decompose()
pheadt = soup.find('div', class_='page-header__top')
pheadt.decompose()
item1 = soup.find('div', class_='global-explore-navigation')
item1.decompose()
navtop = soup.find('div', class_='community-navigation fandom-sticky-header')
navtop.decompose()
ads = soup.find('div', class_='bottom-ads-container')
ads.decompose()
navi = soup.find('nav', class_='global-top-navigation')
navi.decompose()
act = soup.find('div', class_='page-header__actions')
act.decompose()
soup.find_all('table')[1].decompose()
[svg.decompose() for svg in soup.find_all('svg')]
soup.find('div', class_='toc').decompose()

with open(F"test/test.html", "w", encoding = 'utf-8') as file:
    file.write(str(soup.prettify()))